**Objetivo del proyecto:**
Desarrollar un modelo de Machine Learning capaz de identificar las señales de deterioro de los discos de almacenamiento de un centro de datos y estimar la probabilidad de que fallen en los próximos 30 días, permitiendo actuar antes de que suceda.

## 1. Comprensión del problema y elección de la métrica

*El punto de partida es facilitar al equipo de operaciones de un centro de datos saber qué discos tienen una probabilidad alta de fallar pronto, para cambiarlos antes de que provoquen una avería. Target: ¿falla el disco en los próximos 30 días?*

*La respuesta es una categoría con dos valores (1 = falla, 0 = no falla): es un problema de **clasificación binaria**.*

*El error más costoso es el **falso negativo**: que el modelo diga «sano» y el disco falle. Provoca una avería por sorpresa y puede detener el servicio (una caída grave cuesta más de 100.000 $, según el Uptime Institute 2026). El **falso positivo** (revisar un disco que estaba sano) es más barato.*

*Por eso la métrica principal es el **recall de la clase 1** (de todos los discos que fallan, cuántos se detectan). Se tiene en cuenta también la **precisión** (de todas las alarmas, cuántas son reales) para no saturar al equipo con falsas alarmas.*

*Las clases no estarán equilibradas: en una flota real falla en torno al 1-2 % de los discos al año. Por eso:*
- *la **accuracy no sirve** (decir siempre «sano» acertaría casi siempre sin detectar ningún fallo);*
- *para comparar modelos se usa **PR-AUC** (`average_precision`), que resume precisión y recall sin depender del umbral y es la recomendada cuando la clase positiva es rara (documentación de scikit-learn, 3.4 Metrics and scoring).*

*La predicción se traducirá en dos acciones con costes distintos: **vigilar** un disco (revisarlo y comprobar las copias) es barato, así que ahí prima detectar muchos fallos (recall); **sustituirlo** cuesta dinero, así que ahí prima que la alarma sea real (precisión). Por eso el modelo dará una probabilidad y se elegirán **dos umbrales** con validación cruzada, nunca con el test.*